# 매뉴얼 VQA 검수 (Colab)

관리자 페이지에서 받은 **earlier + current JSON**과 **페이지 이미지 ZIP**으로 QA를 보고 고치거나 지웁니다.

1. **1) 설정** 셀에서 Drive 경로를 맞추고 실행합니다. 그 경로에 파일이 없으면 업로드 창이 뜨고, 올린 파일은 그 경로로 Drive에 저장됩니다. (ZIP은 용량이 커서 Drive에 먼저 올려 두는 편이 빠릅니다.)
2. **2) 검수 화면** 셀을 실행합니다.
3. 고친 뒤 **수정 완료**: 누르는 순간 Drive의 JSON을 다시 읽어 그 QA만 바꾸고 바로 저장합니다.
   **제거**: 두 번 눌러 확인하면 Drive의 JSON에서 그 QA를 지우고 바로 저장합니다.

- 시작할 때 JSON 백업을 같은 폴더의 `backups/`에 한 번 만듭니다. 지운 QA는 `<JSON이름>_deleted.jsonl`에도 남습니다.
- bbox는 웹과 같은 형식(`0~1000` 정규화, `x1, y1, x2, y2`)이고, 칸을 비우면 bbox 없음으로 저장됩니다.

In [ ]:
#@title 1) 설정 · Drive 연결 · 이미지 준비
JSON_PATH = "/content/drive/MyDrive/vqa_review/manual_vqa_all_rounds.json"  #@param {type:"string"}
ZIP_PATH = "/content/drive/MyDrive/vqa_review/vision_primary_gt_images.zip"  #@param {type:"string"}
IMG_DIR = "/content/gt_images"   # ZIP을 풀어 둘 Colab 로컬 폴더

import os, re, json, shutil, zipfile, datetime
from collections import Counter
from google.colab import drive, files

drive.mount("/content/drive")


def ensure_on_drive(path, label):
    """Drive 경로에 없으면 업로드 받아 그 경로에 저장"""
    if os.path.exists(path):
        print(f"✓ {label}: {path}")
        return
    print(f"{label} 파일이 {path} 에 없습니다. 업로드하면 그 경로로 Drive에 저장합니다.")
    up = files.upload()
    if not up:
        raise RuntimeError(f"{label} 업로드가 취소되었습니다")
    name, data = next(iter(up.items()))
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        f.write(data)
    if os.path.exists(name):
        os.remove(name)
    print(f"✓ {label} 저장: {path}")


ensure_on_drive(JSON_PATH, "JSON (earlier+current)")
ensure_on_drive(ZIP_PATH, "이미지 ZIP")


def base(p):
    return str(p or "").replace("\\", "/").rsplit("/", 1)[-1]


def zip_name(info):
    """UTF-8 표시가 없는 ZIP(윈도우에서 만든 한글 이름 등)도 이름이 깨지지 않게"""
    n = info.filename
    if not info.flag_bits & 0x800:
        raw = n.encode("cp437")
        for enc in ("utf-8", "cp949"):
            try:
                return raw.decode(enc)
            except UnicodeDecodeError:
                pass
    return n


IMG_EXT = re.compile(r"\.(png|jpe?g|webp|gif|bmp|tiff?)$", re.I)
os.makedirs(IMG_DIR, exist_ok=True)
n_img = 0
with zipfile.ZipFile(ZIP_PATH) as z:
    for info in z.infolist():
        if info.is_dir():
            continue
        name = base(zip_name(info))
        if not name or name.startswith(".") or not IMG_EXT.search(name):
            continue
        dst = os.path.join(IMG_DIR, name.lower())
        if not os.path.exists(dst):
            with z.open(info) as src, open(dst, "wb") as out:
                shutil.copyfileobj(src, out)
        n_img += 1

IMG_INDEX = {}
for fn in os.listdir(IMG_DIR):
    p = os.path.join(IMG_DIR, fn)
    IMG_INDEX[fn.lower()] = p
    IMG_INDEX.setdefault(os.path.splitext(fn.lower())[0], p)   # 확장자가 달라도 찾도록


def find_image(gt):
    b = base(gt).lower()
    return IMG_INDEX.get(b) or IMG_INDEX.get(os.path.splitext(b)[0])


with open(JSON_PATH, encoding="utf-8") as f:
    rows0 = json.load(f)
if not isinstance(rows0, list):
    raise ValueError("JSON 최상위가 리스트가 아닙니다. 관리자 페이지의 earlier+current 내보내기 파일인지 확인하세요.")

BACKUP_DIR = os.path.join(os.path.dirname(JSON_PATH), "backups")
os.makedirs(BACKUP_DIR, exist_ok=True)
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
backup = os.path.join(BACKUP_DIR, f"{os.path.splitext(os.path.basename(JSON_PATH))[0]}_{stamp}.json")
shutil.copy2(JSON_PATH, backup)

rounds0 = Counter((r.get("round") or "(없음)") for r in rows0)
missing = sorted({base(r.get("gt_image")) for r in rows0 if not find_image(r.get("gt_image"))})
print(f"이미지 {n_img}장 · QA {len(rows0)}개 ({', '.join(f'{k} {v}' for k, v in rounds0.items())})")
print(f"백업: {backup}")
if missing:
    print(f"⚠ ZIP에 없는 이미지 {len(missing)}개: {', '.join(missing[:10])}{' …' if len(missing) > 10 else ''}")

In [ ]:
#@title 2) 검수 화면
import io, html
from collections import Counter
import ipywidgets as W
from IPython.display import display
from PIL import Image, ImageDraw

if int(W.__version__.split(".")[0]) >= 8:
    from google.colab import output
    output.enable_custom_widget_manager()

DELETED_LOG = os.path.splitext(JSON_PATH)[0] + "_deleted.jsonl"


# ---------- Drive JSON: 수정 완료/제거 때마다 다시 읽고 바로 저장 ----------
def read_rows():
    with open(JSON_PATH, encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("JSON 최상위가 리스트가 아닙니다")
    return data


def write_rows(rows):
    text = json.dumps(rows, ensure_ascii=False, indent=2)   # 다 만든 뒤에 한 번에 씀
    with open(JSON_PATH, "w", encoding="utf-8") as f:
        f.write(text)


def now_iso():
    return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")


def who(r):
    a = r.get("annotator") or {}
    return (a.get("email") or a.get("name") or "").strip().lower()


def keyed(rows):
    """행 식별: 편집하지 않는 필드 + 같은 값이 겹칠 때의 순번"""
    seen, out = {}, []
    for r in rows:
        k = (r.get("round") or "", str(r.get("query_id") or ""), base(r.get("gt_image")).lower(),
             str(r.get("qa_index") or ""), who(r))
        n = seen.get(k, 0)
        seen[k] = n + 1
        out.append(k + (n,))
    return out


# ---------- bbox (0~1000 정규화 xyxy) ----------
def parse_bbox(s):
    s = (s or "").strip()
    if not s or s.lower() in ("none", "null", "[]"):
        return None
    nums = re.findall(r"-?\d+(?:\.\d+)?", s)
    if len(nums) != 4:
        raise ValueError("bbox는 숫자 4개 (x1, y1, x2, y2 · 0~1000) 이거나 비워 두세요")
    v = [float(x) for x in nums]
    if any(x < 0 or x > 1000 for x in v):
        raise ValueError("bbox 값은 0~1000 사이여야 합니다")
    x1, x2 = sorted(v[0::2])
    y1, y2 = sorted(v[1::2])
    return [int(x) if x.is_integer() else x for x in (x1, y1, x2, y2)]


def fmt_bbox(b):
    return ", ".join(str(x) for x in b) if isinstance(b, list) and len(b) == 4 else ""


# ---------- 이미지 ----------
_PAGES = {}


def page(path):
    if path not in _PAGES:
        if len(_PAGES) > 30:
            _PAGES.clear()
        _PAGES[path] = Image.open(path).convert("RGB")
    return _PAGES[path]


def render(path, bbox, width):
    im = page(path)
    im = im.resize((width, max(1, round(im.height * width / im.width))), Image.LANCZOS)
    if bbox:
        x1, y1, x2, y2 = bbox
        iw, ih = im.size
        ov = Image.new("RGBA", im.size, (0, 0, 0, 0))
        ImageDraw.Draw(ov).rectangle([x1 * iw / 1000, y1 * ih / 1000, x2 * iw / 1000, y2 * ih / 1000],
                                     fill=(255, 64, 64, 45), outline=(220, 30, 30, 255), width=3)
        im = Image.alpha_composite(im.convert("RGBA"), ov).convert("RGB")
    buf = io.BytesIO()
    im.save(buf, "PNG")
    return buf.getvalue()


# ---------- 위젯 ----------
def esc(s):
    return html.escape(str(s if s is not None else ""))


S = {"rows": [], "keys": [], "view": [], "pos": 0, "by_page": {}, "quiet": False, "arm_del": False, "leave": None}

lab = {"description_width": "90px"}
wide = W.Layout(width="100%")
w_round = W.Dropdown(description="라운드", options=[("전체", "*")], value="*", style=lab)
w_annot = W.Dropdown(description="작성자", options=[("전체", "*")], value="*", style=lab)
w_status = W.Dropdown(description="상태", options=[("전체", "*")], value="*", style=lab)
w_search = W.Text(description="검색", placeholder="query·answer·이미지명·query_id", continuous_update=False, style=lab)
w_reload = W.Button(description="Drive에서 다시 읽기", icon="refresh", layout=W.Layout(width="170px"))
w_prev = W.Button(description="◀ 이전", layout=W.Layout(width="90px"))
w_next = W.Button(description="다음 ▶", layout=W.Layout(width="90px"))
w_pos = W.HTML(layout=W.Layout(margin="0 16px"))
w_width = W.IntSlider(description="이미지 폭", value=760, min=400, max=1600, step=20, continuous_update=False, style=lab)
w_pick = W.Dropdown(description="QA", options=[], layout=wide, style=lab)

w_imgmsg = W.HTML()
w_img = W.Image(format="png", layout=W.Layout(display="none"))
w_info = W.HTML()
w_query = W.Textarea(description="query", layout=W.Layout(width="100%", height="70px"), style=lab)
w_answer = W.Textarea(description="answer", layout=W.Layout(width="100%", height="90px"), style=lab)
w_text = W.Textarea(description="evidence text", layout=W.Layout(width="100%", height="110px"), style=lab)
w_desc = W.Textarea(description="image desc", layout=W.Layout(width="100%", height="90px"), style=lab)
w_bbox = W.Text(description="bbox", placeholder="x1, y1, x2, y2 (0~1000) · 비우면 없음", continuous_update=False, layout=wide, style=lab)
w_save = W.Button(description="수정 완료", button_style="success", icon="check")
w_revert = W.Button(description="되돌리기", icon="undo")
w_del = W.Button(description="제거", button_style="danger", icon="trash")
w_msg = W.HTML()
w_same = W.HTML()
FIELDS = (w_query, w_answer, w_text, w_desc, w_bbox)


def note(text, kind="info"):
    color = {"ok": "#188038", "err": "#d93025", "info": "#5f6368"}[kind]
    w_msg.value = f"<div style='color:{color};padding:4px 0'>{text}</div>"


def form_of(r):
    ev = r.get("evidence") or {}
    return (r.get("query") or "", r.get("answer") or "", ev.get("text") or "",
            ev.get("image_description") or "", fmt_bbox(ev.get("bbox")))


def cur():
    return S["view"][S["pos"]] if S["view"] else None


def dirty():
    i = cur()
    return i is not None and tuple(w.value for w in FIELDS) != form_of(S["rows"][i])


def label(p, i):
    r = S["rows"][i]
    q = " ".join(str(r.get("query") or "").split())
    return f"{p + 1}. [{r.get('round') or '-'}] {r.get('query_id') or ''} · {q[:70]}"


def set_options(w, opts):
    old = w.value
    w.options = opts
    vals = [v for _, v in opts]
    w.value = old if old in vals else vals[0]


def sync_pick():
    S["quiet"] = True
    try:
        w_pick.value = S["pos"] if S["view"] else None
    finally:
        S["quiet"] = False


def set_rows(rows, keep_key=None, keep_pos=0):
    S["rows"], S["keys"] = rows, keyed(rows)
    S["by_page"] = {}
    for i, r in enumerate(rows):
        S["by_page"].setdefault(base(r.get("gt_image")).lower(), []).append(i)
    rounds = Counter(r.get("round") or "" for r in rows)
    statuses = Counter(r.get("status") or "" for r in rows)
    people, names = Counter(who(r) for r in rows), {}
    for r in rows:
        a = r.get("annotator") or {}
        names.setdefault(who(r), f"{a.get('name') or '?'} <{a.get('email')}>" if a.get("email") else (a.get("name") or "(이름 없음)"))
    S["quiet"] = True
    try:
        set_options(w_round, [("전체", "*")] + [(f"{k or '(없음)'} · {n}", k) for k, n in sorted(rounds.items())])
        set_options(w_status, [("전체", "*")] + [(f"{k or '(없음)'} · {n}", k) for k, n in sorted(statuses.items())])
        set_options(w_annot, [("전체", "*")] + sorted(((f"{names[k]} · {n}", k) for k, n in people.items()),
                                                      key=lambda t: t[0].lower()))
    finally:
        S["quiet"] = False
    apply_filter(keep_key, keep_pos)


def apply_filter(keep_key=None, keep_pos=0):
    q = w_search.value.strip().lower()
    view = []
    for i, r in enumerate(S["rows"]):
        if w_round.value != "*" and (r.get("round") or "") != w_round.value:
            continue
        if w_status.value != "*" and (r.get("status") or "") != w_status.value:
            continue
        if w_annot.value != "*" and who(r) != w_annot.value:
            continue
        if q:
            ev = r.get("evidence") or {}
            hay = " ".join(str(x or "") for x in (r.get("query_id"), r.get("gt_image"), r.get("query"),
                                                   r.get("answer"), ev.get("text"), ev.get("image_description")))
            if q not in hay.lower():
                continue
        view.append(i)
    pos = min(keep_pos, len(view) - 1) if view else 0
    if keep_key is not None:
        pos = next((p for p, i in enumerate(view) if S["keys"][i] == keep_key), pos)
    S["view"], S["pos"], S["leave"] = view, max(pos, 0), None
    S["quiet"] = True
    try:
        w_pick.options = [(label(p, i), p) for p, i in enumerate(view)]
    finally:
        S["quiet"] = False
    sync_pick()
    show()


def show():
    S["arm_del"] = False
    w_del.description = "제거"
    w_save.description = "수정 완료"
    i = cur()
    S["quiet"] = True
    try:
        for w, v in zip(FIELDS, form_of(S["rows"][i]) if i is not None else ("",) * 5):
            w.value = v
    finally:
        S["quiet"] = False
    if i is None:
        w_pos.value = "0 / 0"
        w_info.value = "<i>조건에 맞는 QA가 없습니다.</i>"
        w_same.value = w_imgmsg.value = ""
        w_img.layout.display = "none"
        return
    r = S["rows"][i]
    a = r.get("annotator") or {}
    rd = r.get("round") or "-"
    w_pos.value = f"<b>{S['pos'] + 1}</b> / {len(S['view'])}"
    w_info.value = (
        f"<div style='line-height:1.7'><b style='font-size:15px'>{esc(r.get('query_id'))}</b>"
        f" &nbsp;<span style='padding:1px 8px;border-radius:9px;color:#202124;"
        f"background:{'#d2e3fc' if rd == 'current' else '#e8eaed'}'>{esc(rd)}</span>"
        f" &nbsp;QA {esc(r.get('qa_index'))}<br>"
        f"이미지 <code>{esc(base(r.get('gt_image')))}</code> · {esc(r.get('brand'))} · {esc(r.get('source'))}<br>"
        f"작성자 {esc(a.get('name'))} &lt;{esc(a.get('email'))}&gt; · {esc(r.get('status'))}"
        f" · 수정 {esc(r.get('updated_at') or '-')}</div>")
    others = [j for j in S["by_page"].get(base(r.get("gt_image")).lower(), []) if j != i]
    if others:
        items = "".join(f"<li>[{esc(S['rows'][j].get('round') or '-')}] QA {esc(S['rows'][j].get('qa_index'))}"
                        f" · {esc(S['rows'][j].get('query'))}</li>" for j in others)
        w_same.value = (f"<div style='margin-top:6px'><b>같은 페이지의 다른 QA {len(others)}개</b>"
                        f"<ol style='margin:4px 0 0 18px;padding:0'>{items}</ol></div>")
    else:
        w_same.value = ""
    draw()


def draw():
    i = cur()
    if i is None:
        return
    gt = S["rows"][i].get("gt_image")
    path = find_image(gt)
    if not path:
        w_img.layout.display = "none"
        w_imgmsg.value = f"<div style='color:#d93025'>ZIP에 이 이미지가 없습니다: <code>{esc(base(gt))}</code></div>"
        return
    try:
        bb = parse_bbox(w_bbox.value)
        w_imgmsg.value = f"<code>{esc(os.path.basename(path))}</code>" + (f" · bbox {esc(fmt_bbox(bb))}" if bb else " · bbox 없음")
    except ValueError as e:
        bb = None
        w_imgmsg.value = f"<span style='color:#d93025'>{esc(e)}</span>"
    w_img.value = render(path, bb, w_width.value)
    w_img.layout.display = None


def go(p):
    if not S["view"]:
        return
    p = max(0, min(p, len(S["view"]) - 1))
    if p == S["pos"]:
        return sync_pick()
    if dirty() and S["leave"] != p:
        S["leave"] = p
        sync_pick()
        return note("저장 안 한 수정이 있습니다. <b>수정 완료</b>를 누르거나, 버리고 이동하려면 한 번 더 누르세요.", "err")
    S["leave"] = None
    S["pos"] = p
    sync_pick()
    w_msg.value = ""
    show()


# ---------- 버튼 ----------
def on_save(_):
    i = cur()
    if i is None:
        return
    key = S["keys"][i]
    try:
        bb = parse_bbox(w_bbox.value)
    except ValueError as e:
        return note(esc(e), "err")
    try:
        fresh = read_rows()                      # 누를 때마다 Drive 파일을 새로 읽음
        ks = keyed(fresh)
        if key not in ks:
            set_rows(fresh, keep_pos=S["pos"])
            return note("Drive 파일에 이 QA가 없습니다 (다른 곳에서 지워졌을 수 있음). 최신 파일로 다시 읽었습니다.", "err")
        r = fresh[ks.index(key)]
        r["query"] = w_query.value.strip()
        r["answer"] = w_answer.value.strip()
        ev = dict(r.get("evidence") or {})
        ev["text"] = w_text.value.strip()
        ev["image_description"] = w_desc.value.strip()
        ev["bbox"] = bb
        ev.setdefault("bbox_format", "normalized_0_1000_xyxy")
        r["evidence"] = ev
        r["status"] = "done"                     # 검수하고 저장한 QA는 완료로 (draft도)
        r["updated_at"] = now_iso()
        write_rows(fresh)
    except Exception as e:
        return note(f"저장 실패: {esc(e)}", "err")
    set_rows(fresh, keep_key=key, keep_pos=S["pos"])
    note(f"✓ 저장됨 · {esc(r.get('query_id'))} · Drive JSON 갱신 (QA {len(fresh)}개)", "ok")


def on_delete(_):
    i = cur()
    if i is None:
        return
    if not S["arm_del"]:
        S["arm_del"] = True
        w_del.description = "정말 제거? 한 번 더"
        return note("제거하려면 <b>제거</b>를 한 번 더 누르세요.", "err")
    key, pos = S["keys"][i], S["pos"]
    try:
        fresh = read_rows()                      # 누를 때마다 Drive 파일을 새로 읽음
        ks = keyed(fresh)
        if key not in ks:
            set_rows(fresh, keep_pos=pos)
            return note("Drive 파일에 이미 없는 QA입니다. 최신 파일로 다시 읽었습니다.", "err")
        gone = fresh.pop(ks.index(key))
        with open(DELETED_LOG, "a", encoding="utf-8") as f:   # 잘못 지웠을 때 되살릴 사본
            f.write(json.dumps({"deleted_at": now_iso(), "row": gone}, ensure_ascii=False) + "\n")
        write_rows(fresh)
    except Exception as e:
        return note(f"제거 실패: {esc(e)}", "err")
    set_rows(fresh, keep_pos=pos)
    note(f"✓ 제거됨 · {esc(gone.get('query_id'))} · Drive JSON 갱신 (QA {len(fresh)}개). "
         f"지운 QA 사본: <code>{esc(os.path.basename(DELETED_LOG))}</code>", "ok")


def on_reload(_):
    i = cur()
    try:
        set_rows(read_rows(), keep_key=S["keys"][i] if i is not None else None, keep_pos=S["pos"])
        note(f"Drive에서 다시 읽음 · QA {len(S['rows'])}개", "ok")
    except Exception as e:
        note(f"읽기 실패: {esc(e)}", "err")


def on_revert(_):
    show()
    note("저장된 값으로 되돌렸습니다.")


def on_pick(ch):
    if not S["quiet"] and ch["new"] is not None:
        go(ch["new"])


def on_filter(ch):
    if not S["quiet"]:
        i = cur()
        apply_filter(keep_key=S["keys"][i] if i is not None else None)


def on_field(ch):
    if S["quiet"]:
        return
    w_save.description = "수정 완료 *" if dirty() else "수정 완료"
    if ch["owner"] is w_bbox:
        draw()


w_save.on_click(on_save)
w_del.on_click(on_delete)
w_reload.on_click(on_reload)
w_revert.on_click(on_revert)
w_prev.on_click(lambda _: go(S["pos"] - 1))
w_next.on_click(lambda _: go(S["pos"] + 1))
w_pick.observe(on_pick, names="value")
for w in (w_round, w_status, w_annot, w_search):
    w.observe(on_filter, names="value")
for w in FIELDS:
    w.observe(on_field, names="value")
w_width.observe(lambda ch: draw(), names="value")

left = W.VBox([w_imgmsg, w_img], layout=W.Layout(width="58%", overflow="auto", max_height="1100px"))
right = W.VBox([w_info, w_query, w_answer, w_text, w_desc, w_bbox,
                W.HBox([w_save, w_revert, w_del]), w_msg, w_same],
               layout=W.Layout(width="42%", padding="0 0 0 12px"))
display(W.VBox([
    W.HBox([w_round, w_status, w_annot, w_search, w_reload], layout=W.Layout(flex_flow="row wrap")),
    W.HBox([w_prev, w_next, w_pos, w_width], layout=W.Layout(align_items="center")),
    w_pick,
    W.HBox([left, right]),
]))
set_rows(read_rows())

In [ ]:
#@title 3) (선택) 작업 끝 — Drive에 바로 반영하고 연결 해제
drive.flush_and_unmount()
print("Drive 동기화 완료. 다시 쓰려면 1)부터 실행하세요.")